# Terra Signal: geração do arquivo de previsão (Grupo 1)

Treina o modelo escolhido no notebook 09 e gera o arquivo `prediction_<equipe>_<timestamp>.csv` para o `inference.csv`.

Cada pergunta tem três células: **pergunta** → **código** → **resposta**.

- **Modelo:** regressão logística com C = 1 e as 25 dummies do notebook 07, sem `CustomerFeedback`.
- **O corte de 0,5 é provisório.** A TS ainda não informou a métrica de avaliação nem os custos da ação. Para trocar o corte, basta mudar `CORTE` e executar de novo.
- **Cada execução cria um novo arquivo**, com outro timestamp. O formato segue o `NOTEBOOK.ipynb` do repositório.
- Roda com pandas, numpy, scipy e scikit-learn, sem Spark. Funciona localmente e no Databricks (Git folder).

## Setup

In [1]:
import datetime
import os
import warnings

import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

CORTE = 0.5  # provisório
NOME_EQUIPE = "Grupo1"
C_FINAL = 1  # escolhido na pergunta 8 do notebook 09

DATA_DIR = "../data/raw/"  # este notebook fica em notebooks/; os CSVs ficam em data/raw/
OUT_DIR = "../data/processed/"  # onde o arquivo de previsão é salvo
os.makedirs(OUT_DIR, exist_ok=True)
hist_raw = pd.read_csv(DATA_DIR + "history.csv")
inf_raw = pd.read_csv(DATA_DIR + "inference.csv")
print("history:", hist_raw.shape, "| inference:", inf_raw.shape)

history: (5634, 23) | inference: (1409, 22)


In [2]:
def prep(d):
    """Regras do notebook 06: recupera tenure. TotalCharges só serve para isso e não entra no modelo."""
    d = d.copy()
    d["tenure_num"] = pd.to_numeric(d["tenure"], errors="coerce")
    d["tenure_num"] = d["tenure_num"].fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
    return d


hist = prep(hist_raw)
hist = hist[hist["tenure_num"].notna()].reset_index(drop=True)  # 1 linha sem tenure e sem TotalCharges
inf = prep(inf_raw)
y = (hist["Churn"] == "Yes").astype(int)
print("Clientes no treino:", len(hist), "| churners:", y.sum(), f"({y.mean():.1%})")

Clientes no treino: 5633 | churners: 1503 (26.7%)


In [3]:
def melhor_corte(x, y, min_folha):
    """Corte binário de maior qui-quadrado, com p-valor corrigido por Bonferroni pelo nº de cortes testados."""
    tab = pd.DataFrame({"x": x, "y": y}).groupby("x")["y"].agg(["size", "sum"]).sort_index()
    n_esq, c_esq = tab["size"].cumsum().values, tab["sum"].cumsum().values
    n, c = n_esq[-1], c_esq[-1]
    validos = (n_esq >= min_folha) & (n - n_esq >= min_folha)
    if not validos.any():
        return None
    a, b = c_esq, n_esq - c_esq
    e, f = c - c_esq, (n - n_esq) - (c - c_esq)
    den = (a + b) * (e + f) * (a + e) * (b + f)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = np.where(den > 0, n * (a * f - b * e) ** 2 / den, 0.0)
    q = np.where(validos, q, -1.0)
    i = int(np.argmax(q))
    return float(tab.index[i]), float(q[i]), min(1.0, chi2.sf(q[i], 1) * int(validos.sum()))


def aid(x, y, min_folha=0.05, alfa=0.05, max_faixas=None):
    """AID do notebook 07: divide sempre a faixa com maior qui-quadrado."""
    x, y = np.asarray(x, float), np.asarray(y)
    tam = int(np.ceil(min_folha * len(x)))
    nos, cortes = [np.ones(len(x), bool)], []
    while not max_faixas or len(nos) < max_faixas:
        candidatos = []
        for k, m in enumerate(nos):
            r = melhor_corte(x[m], y[m], tam)
            if r and r[2] < alfa:
                candidatos.append((r[1], k, r))
        if not candidatos:
            break
        _, k, r = max(candidatos, key=lambda z: z[0])
        m = nos.pop(k)
        nos += [m & (x <= r[0]), m & (x > r[0])]
        cortes.append(r[0])
    return sorted(cortes)


DUMMIES_CATEG = {
    "gender": ["Female"],
    "SeniorCitizen": [1],
    "Partner": ["Yes"],
    "Dependents": ["Yes"],
    "MultipleLines": ["Yes", "No phone service"],
    "InternetService": ["DSL", "No"],
    "OnlineSecurity": ["Yes"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["Yes"],
    "TechSupport": ["Yes"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["One year", "Two year"],
    "PaperlessBilling": ["No"],
    "PaymentMethod": ["Mailed check", "Credit card (automatic)", "Bank transfer (automatic)"],
}


def dummies_faixas(x, cortes, ref):
    """Uma dummy por faixa de tenure, exceto a faixa de referência `ref`."""
    f = np.digitize(x, cortes, right=True)
    limites = [-np.inf] + list(cortes) + [np.inf]
    return pd.DataFrame(
        {f"tenure=({limites[i]:g}, {limites[i + 1]:g}]": (f == i).astype(int) for i in range(len(limites) - 1) if i != ref},
        index=x.index,
    )


def montar_dummies(d, cortes, ref):
    cat = pd.DataFrame(
        {f"{col}={val}": (d[col] == val).astype(int) for col, vals in DUMMIES_CATEG.items() for val in vals},
        index=d.index,
    )
    return pd.concat([cat, dummies_faixas(d["tenure_num"], cortes, ref)], axis=1)


def faixa_referencia(x, cortes):
    return int(pd.Series(np.digitize(x, cortes, right=True)).value_counts().idxmax())

### 1. O que esperar desse corte fora da amostra?

In [4]:
linhas = []
for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0).split(hist, y):
    cortes = aid(hist["tenure_num"].iloc[tr], y.iloc[tr], max_faixas=6)
    ref = faixa_referencia(hist["tenure_num"].iloc[tr], cortes)
    m = LogisticRegression(C=C_FINAL, max_iter=3000).fit(montar_dummies(hist.iloc[tr], cortes, ref), y.iloc[tr])
    prev = m.predict_proba(montar_dummies(hist.iloc[te], cortes, ref))[:, 1] >= CORTE
    yte = y.iloc[te]
    linhas.append({"acuracia": accuracy_score(yte, prev), "precisao": precision_score(yte, prev), "recall": recall_score(yte, prev),
                   "F1": f1_score(yte, prev), "pct_yes": prev.mean()})
print(pd.DataFrame(linhas).agg(["mean", "std"]).round(4).to_string())

      acuracia  precisao  recall      F1  pct_yes
mean    0.8013    0.6637  0.5181  0.5814   0.2083
std     0.0089    0.0191  0.0350  0.0256   0.0131


**Resposta:** com corte em 0,5, a acurácia esperada é de cerca de **80,1%**, contra 73,3% de quem prevê sempre `No`. Dos clientes marcados como `Yes`, 66,4% de fato são churners (precisão), e o modelo encontra 51,8% dos churners (recall), ou seja, deixa passar quase metade. O F1 é de 0,581.

### 2. Quantos clientes do `inference` ficam marcados como `Yes`?

In [5]:
cortes_final = aid(hist["tenure_num"], y, max_faixas=6)
ref_final = faixa_referencia(hist["tenure_num"], cortes_final)
modelo = LogisticRegression(C=C_FINAL, max_iter=3000).fit(montar_dummies(hist, cortes_final, ref_final), y)
p_inf = modelo.predict_proba(montar_dummies(inf, cortes_final, ref_final))[:, 1]
print("Cortes de tenure:", cortes_final)
print(f"Yes: {(p_inf >= CORTE).sum()} de {len(p_inf)} ({(p_inf >= CORTE).mean():.1%})")
print(f"Probabilidade média: {p_inf.mean():.1%} | soma das probabilidades: {p_inf.sum():.0f} clientes")

Cortes de tenure: [5.0, 17.0, 43.0, 59.0, 70.0]
Yes: 297 de 1409 (21.1%)
Probabilidade média: 26.5% | soma das probabilidades: 374 clientes


**Resposta:** 297 de 1.409 clientes (21,1%). A soma das probabilidades indica cerca de 374 churners esperados, então esse corte marca menos clientes do que o total esperado, o que é coerente com o recall da pergunta 1.

### 3. O arquivo gerado está no formato esperado?

In [6]:
prediction = inf[["customerID"]].copy()
prediction["prediction"] = np.where(p_inf >= CORTE, "Yes", "No")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
arquivo = f"prediction_{NOME_EQUIPE}_{timestamp}.csv"
prediction.to_csv(OUT_DIR + arquivo)  # mesmo formato do NOTEBOOK.ipynb

lido = pd.read_csv(OUT_DIR + arquivo, index_col=0)
print("Arquivo:", arquivo)
print("Colunas:", list(lido.columns), "| linhas:", len(lido))
print("Mesmos clientes, na mesma ordem do inference:", lido["customerID"].tolist() == inf_raw["customerID"].tolist())
print("IDs repetidos:", lido["customerID"].duplicated().sum(), "| valores vazios:", int(lido.isna().sum().sum()))
print("Valores:", lido["prediction"].value_counts().to_dict())
print(lido.head().to_string())

Arquivo: prediction_Grupo1_20261112_213829.csv
Colunas: ['customerID', 'prediction'] | linhas: 1409
Mesmos clientes, na mesma ordem do inference: True
IDs repetidos: 0 | valores vazios: 0
Valores: {'No': 1112, 'Yes': 297}
   customerID prediction
0  4226-KKDON         No
1  8838-GPHZP         No
2  0320-JDNQG         No
3  4837-PZTIC         No
4  7748-UMTRK        Yes


**Resposta:** sim. O arquivo tem as colunas `customerID` e `prediction`, os mesmos clientes e a mesma ordem do `inference.csv`, sem repetições nem vazios, e só os valores `Yes` e `No`.

### 4. O arquivo de scores do app está no formato que ele lê?

O app (`app/app.py`) lê `data/processed/terra_signal_churn_scores.csv` com as colunas `customerID`, `ChurnScore`, `ChurnPredicted` e `Rank`. Aqui o `ChurnScore` é a probabilidade calibrada da regressão logística. Também entram duas colunas da regra de decisão do notebook 12, no cenário base: `LucroEsperado` (R$ em 12 meses) e `Ligar`.

In [7]:
# Premissas do cenário base (notebook 12)
DELTA_A, W, U, D, C_LIG = 0.20, 0.01, 0.30, 60, 10

V = 12 * inf["MonthlyCharges"].to_numpy()
lucro = p_inf * DELTA_A * (V - D) - (1 - p_inf) * (U * D + W * V) - C_LIG

scores = pd.DataFrame({
    "customerID": inf["customerID"],
    "ChurnScore": p_inf.round(4),
    "ChurnPredicted": np.where(p_inf >= CORTE, "Yes", "No"),  # mesmo corte do arquivo de entrega
    "Rank": pd.Series(p_inf).rank(ascending=False, method="first").astype(int).to_numpy(),
    "LucroEsperado": lucro.round(2),
    "Ligar": np.where(lucro > 0, "Sim", "Não"),
})
scores.to_csv(OUT_DIR + "terra_signal_churn_scores.csv", index=False, encoding="utf-8-sig")

# Lê do mesmo jeito que o app
lido = pd.read_csv(OUT_DIR + "terra_signal_churn_scores.csv", sep=",", decimal=".", encoding="utf-8-sig")
print("Colunas:", list(lido.columns), "| linhas:", len(lido), "| IDs repetidos:", lido["customerID"].duplicated().sum())
print("ChurnPredicted:", lido["ChurnPredicted"].value_counts().to_dict(), "| Ligar:", lido["Ligar"].value_counts().to_dict())
print(f"ChurnScore de {lido['ChurnScore'].min():.3f} a {lido['ChurnScore'].max():.3f} | lucro esperado da lista: R$ {lido.loc[lido['Ligar'] == 'Sim', 'LucroEsperado'].sum():,.0f}")
print(lido.sort_values("Rank").head().to_string(index=False))


Colunas: ['customerID', 'ChurnScore', 'ChurnPredicted', 'Rank', 'LucroEsperado', 'Ligar'] | linhas: 1409 | IDs repetidos: 0
ChurnPredicted: {'No': 1112, 'Yes': 297} | Ligar: {'Não': 775, 'Sim': 634}
ChurnScore de 0.003 a 0.907 | lucro esperado da lista: R$ 40,659
customerID  ChurnScore ChurnPredicted  Rank  LucroEsperado Ligar
1374-DMZUI      0.9072            Yes     1         181.71   Sim
9300-AGZNL      0.9058            Yes     2         180.71   Sim
5150-ITWWB      0.9058            Yes     3         182.55   Sim
6630-UJZMY      0.8861            Yes     4         153.21   Sim
4910-GMJOT      0.8832            Yes     5         176.49   Sim


**Resposta:** sim. O arquivo tem os 1.409 clientes do `inference`, sem IDs repetidos, e o app lê as quatro colunas que usa. `ChurnPredicted` segue o corte de 0,5 do arquivo de entrega (297 `Yes`). `Ligar` segue a regra de lucro e marca 634 clientes, com cerca de R$ 41 mil de lucro esperado, os mesmos números do pitch. O app ordena pelo `ChurnScore`; quem quiser a lista de ligações deve filtrar `Ligar = Sim`, porque a regra também considera a mensalidade.

## Limites desta análise

- **O corte é provisório:** ele não é o ponto ideal para o negócio, que depende dos custos da ação e da taxa de aceite. Um corte menor encontra mais churners, em troca de mais falsos alarmes.
- **O formato segue o exemplo do repositório:** o arquivo inclui a coluna de índice que o `NOTEBOOK.ipynb` gera por padrão, porque a organização não publicou outra especificação.
- **O desempenho vem do `history`:** o `inference.csv` não tem `Churn`, então não dá para medir a nota real deste arquivo. As decisões anteriores usaram a mesma base, então o resultado real pode ser um pouco menor.
- **Clientes do `inference`:** falta a TS confirmar se todos estão ativos.